# Notebook 05 — Wealth signals: derive, don't insert

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

The referral screen shows *wealth signals* — the evidence a banker uses to decide a referral.
Where do those signals come from, and why can we trust them? Could we just write the ones the
mockup shows?

This notebook answers that. It is the first Workshop 2 notebook about how signals are *derived*,
and it teaches the single most important rule in the architecture: **a signal is derived from
real data through a rule and validated before it is written — never hand-inserted to make a
screen look complete.** That rule (the SR 11-7 / OCC 2011-12 model-risk discipline) is what makes
the signals defensible evidence rather than decoration.

## Key terms for this notebook

| Term | Meaning |
|------|---------|
| **WealthSignal** | A typed, dated (where applicable) observation that a customer may be wealth-eligible. Attached to a customer by `atlas:producesSignal`. |
| **Derivation** | A SPARQL `CONSTRUCT` that reads real facts (transactions, coverage) and *constructs* signal triples from them. The signal exists because the rule fired, not because someone typed it. |
| **Validate-before-write** | The constructed triples are checked against the SHACL shape (`atlas:WealthSignalTypeShape`) with pyshacl; they are written only if they conform. Non-conformance halts the write. |
| **derive-don't-insert** | The non-negotiable discipline: signals emerge from the graph; they are never hand-placed. A signal the data cannot support is *not produced* — the absence is honest. |

Phase 1 honestly derives **two** signal types for the demo household: **Large Deposit Pattern**
(Workshop 1) and **No Advisor Coverage** (Workshop 2). A third the mockup shows — **Segment Shift**
— is *not* derivable from Phase-1 data, and the last section explains why refusing to fake it is
the lesson, not a gap.

## The concept: a signal is a claim the data has to earn

A wealth signal is a structured assertion — "this customer shows behaviour consistent with wealth
readiness." In a regulated institution that assertion has to be *explainable*: a reviewer must be
able to point at the in-bank facts that produced it and the rule that fired. That is why every
signal in ATLAS is the output of a deterministic `CONSTRUCT` over promoted data, validated against
a SHACL shape before it lands in the Semantic Layer Graph Database (SLGD).

This has a sharp consequence. If the data does not contain the facts a signal's definition
requires, the rule does not fire, and **the signal does not exist** — we do not write it anyway to
match a design. A fabricated signal is worse than a missing one: it is an unexplainable assertion
in a system whose entire purpose is explainability. The two signals below are derived honestly;
the third is honestly absent.

The attachment predicate is always `atlas:producesSignal` (Customer → WealthSignal), and every
derived signal carries `prov:wasGeneratedBy <…signal-derivation-run>` so its provenance — and its
removal — is auditable.

In [ ]:
# WS2 notebook setup — installs only what this notebook needs into the kernel.
import sys, subprocess

pkgs = ['rdflib>=7.0.0', 'pyshacl>=0.25.0']
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check'] + pkgs,
    cwd='/tmp'
)
print('Dependencies ready.')

## Build 1 — Large Deposit Pattern (Workshop 1 owns this; we read, we don't re-write)

`LargeDepositPattern` is derived **in Workshop 1**:
`agentic-semantic-layer/notebooks/05_entity_resolution.ipynb`, cell `cell-09f-derive-signals-live`
runs the authoritative `CONSTRUCT` and the pyshacl validate-before-write that persists it. Workshop 2
does **not** re-derive or re-write it — that would fork the source of truth.

The rule: a customer with a `DEPOSIT` ≥ \$250,000 in the observation window **and** no active
advisory coverage produces a `LargeDepositPattern` signal, evidenced by the transaction.

To *see the mechanism* without forking it, the next cell runs the WHERE pattern **read-only** —
it `SELECT`s the customers the rule would match. It performs no `CONSTRUCT` write and no `INSERT`.
The authoritative derivation stays in WS1.

In [ ]:
# READ-ONLY demonstration of the WS1 LargeDepositPattern rule. No INSERT, no CONSTRUCT-write.
# This SELECT mirrors the WHERE clause of WS1 nb05 cell-09f purely to show which customers the
# rule matches. The authoritative signal is derived and written by Workshop 1, not here.
LDP_RULE_READONLY = '''
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT DISTINCT ?customer ?txn ?amount ?txnDate WHERE {
    ?customer a atlas:Customer ; atlas:hasAccount ?account .
    ?account atlas:hasTransaction ?txn .
    ?txn atlas:amountUSD ?amount ; atlas:transactionDate ?txnDate ;
         atlas:transactionType "DEPOSIT"^^xsd:string .
    FILTER (?amount >= 250000)
    FILTER NOT EXISTS {
        ?customer atlas:hasAdvisor ?rel . ?rel a atlas:AdvisoryRelationship .
        FILTER NOT EXISTS { ?rel atlas:coverageEndDate ?end }
    }
}'''
# Run against the live SLGD via the same read path the resolver uses (banker JWT -> sparql MCP).
# Verification cell below shows the expected shape; this string documents the exact rule.
print('LargeDepositPattern rule (read-only). Authoritative write: WS1 nb05 cell-09f.')
print('Emitted by WS1 (NOT here): ?signal a atlas:WealthSignal ; atlas:hasSignalType atlas:LargeDepositPattern ;')
print('                            atlas:signalDate ?txnDate ; atlas:evidencedBy ?txn ;')
print('                            prov:wasGeneratedBy <inst:signal-derivation-run> . ?customer atlas:producesSignal ?signal .')

## Build 2 — No Advisor Coverage (the Workshop 2 native derivation, end to end)

This is the signal Workshop 2 derives itself, so we teach the full cycle:
`use-case-applications/scripts/derive-no-advisor-coverage.py`.

**Gate C — why a gate, not a census.** "No advisor coverage" alone fires for ~120 of 200 customers
(60%) — that is a *list of the uncovered*, not a *signal*. We fire it only for a customer who is
**already wealth-signalled** (produces another `WealthSignal`) **and** has no active coverage. That
is a coverage *gap that matters* — a wealth-eligible customer a referral could serve. 40 customers
qualify (including the demo customer c6b6e4ad).

**The absence-signal shape.** A coverage gap has no positive evidence transaction and no event date,
so the signal carries **only** `hasSignalType` + `producesSignal` + the provenance stamp — **no
`evidencedBy`, no `signalDate`**. The SHACL shape (`atlas:WealthSignalTypeShape`) requires exactly
one `hasSignalType` and nothing else, so an evidence-less signal conforms honestly. We do **not**
invent a fake transaction or date to fill the template.

**Validate-before-write.** `validate_signals()` runs pyshacl against Workshop 1's `atlas-shapes.ttl`
and returns the triples *only if* the candidate graph conforms — otherwise it raises and nothing is
written. The caller inserts only what validated.

In [ ]:
# The gate-C rule (read-only SELECT of who qualifies), mirroring derive-no-advisor-coverage.py.
# The script's CONSTRUCT emits the absence-signal triples; validate_signals() is the pyshacl gate.
NAC_GATE_C_READONLY = '''
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT DISTINCT ?customer WHERE {
    ?customer a atlas:Customer ; atlas:producesSignal ?anySig .   # gate C: already wealth-signalled
    FILTER NOT EXISTS {                                            # AND no active coverage
        ?customer atlas:hasAdvisor ?rel .
        FILTER NOT EXISTS { ?rel atlas:coverageEndDate ?end }
    }
}'''
print('NoAdvisorCoverage gate C — emitted triples (absence shape; no evidencedBy/signalDate):')
print('  ?signal a atlas:WealthSignal ; atlas:hasSignalType atlas-part-2:NoAdvisorCoverageSignal ;')
print('          prov:wasGeneratedBy <inst:signal-derivation-run> . ?customer atlas:producesSignal ?signal .')
print('Validated by pyshacl vs atlas:WealthSignalTypeShape before INSERT (validate_signals()).')

## Build 3 — Segment Shift: the signal we refuse to fake

The mockup shows a third tile: **"Segment shift — Mass-affluent → HNW threshold crossed."** The
concept exists and is loaded (`atlas-part-2:SegmentShiftSignal`). But Phase-1 data **cannot fire it
honestly**, for concrete reasons:

1. **No segment/tier model.** The SLGD has no customer-segment concept or band thresholds at all —
   there is nothing that says a customer *is* mass-affluent or HNW.
2. **No temporal dimension.** Accounts carry only a *current* `balanceUSD` — there is no prior
   balance, no dated snapshot, no history. A "shift" is a claim about **change over time**, and the
   data has no over-time.
3. **The demo customer doesn't cross.** c6b6e4ad's current total balance is ~\$350k (mass-affluent,
   well below the \$1M HNW line). Its only large event is the \$1.8M deposit on 2026-03-03 — which
   is *already* the evidence for its Large Deposit Pattern signal. Re-using it as a "segment shift"
   would be an **echo** of an existing signal, not a distinct insight.

We could *seed* a fake prior-segment history to make the tile light up. **We don't** — that is
hand-assigning a customer's segment to match a picture, the exact opposite of derive-don't-insert.
A genuine segment-shift signal needs real temporal/behavioural data (the `atlas-part-2:Session`
velocity dimension introduced in the **session-intelligence phase**). Until that data exists, the
honest card shows two signals, and Segment Shift is `enabled: false` in `wealth-signals.yaml`.

**This refusal is the lesson.** The same discipline that produced the first two signals is what
forbids the third. A model-risk reviewer can trust the two precisely because we did not manufacture
the third.

## Verification

Run these against the live SLGD (with a banker JWT, the same read path the GraphQL resolver uses).
Each is **read-only** — this notebook never writes signals.

Expected:
- The LDP read-only rule returns the matched customers (incl. c6b6e4ad with its \$1.8M deposit).
- The NAC gate-C rule returns **40** customers (incl. c6b6e4ad).
- The demo household card returns **two** labeled signals: "Large Deposit Pattern" and
  "No Advisor Coverage Signal". No Segment Shift — correctly, because none was derived.

Remediation: if NAC returns 0, the gate read before other signals were persisted (ordering) — see
derive-no-advisor-coverage.py's ordering note. If labels show raw URIs, the atlas-part-2: concepts
are not loaded — run scripts/load-ws2-ontology-concepts.py first.

In [ ]:
# Read-only verification (uses the banker-JWT -> atlas-sparql-mcp read path; no writes).
# Counts only — confirms the rules match real data and the card shows exactly what was derived.
VERIFY = {
    'LDP matches (read-only)': 'SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a atlas:Customer ; atlas:hasAccount ?a . ?a atlas:hasTransaction ?t . ?t atlas:amountUSD ?amt ; atlas:transactionType "DEPOSIT"^^xsd:string . FILTER(?amt >= 250000) }',
    'NAC gate-C fires (expect 40)': 'SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a atlas:Customer ; atlas:producesSignal ?s . FILTER NOT EXISTS { ?c atlas:hasAdvisor ?r . FILTER NOT EXISTS { ?r atlas:coverageEndDate ?e } } }',
    'SegmentShift signals (expect 0 — not derived)': 'SELECT (COUNT(?s) AS ?n) WHERE { ?s atlas:hasSignalType atlas-part-2:SegmentShiftSignal }',
}
for label, q in VERIFY.items():
    print(f'{label}:')
    print(f'  {q[:90]}...')
# Execute these via your SLGD read helper (banker JWT). Counts confirm: rules match, no faked third.
print('\nAll read-only. This notebook derives nothing and writes nothing.')

## What just changed

You now know how Workshop 2's wealth signals come to exist: a `CONSTRUCT` reads real in-bank facts,
pyshacl validates the result against the SHACL shape, and only conformant triples are written —
`atlas:producesSignal` linking the customer to the signal. Large Deposit Pattern is owned and
written by Workshop 1; No Advisor Coverage is derived by Workshop 2's `derive-no-advisor-coverage.py`
with a gate that makes it a *signal* rather than a census.

Most importantly, you saw the discipline refuse a third signal. Segment Shift is real as a concept
but unsupported by Phase-1 data, so it is not produced — not seeded, not relabelled, not faked. The
honest two-signal card is the product of the same rule that would have let us fake three. That rule
— derive-don't-insert, validate-before-write — is what makes these signals defensible evidence under
SR 11-7. The session-intelligence phase, with real temporal data, is where Segment Shift earns its
place honestly.